# 05 · Time, place and the denominator

**45 minutes · Intermediate.** Draw a report-date curve, compare district measures, and explore population sensitivity without claiming causal effects.

[Run this lab in JupyterLite](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/lab/index.html?path=05_Spatial_Epidemiology.ipynb) · [Book home](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/book/intro.html)

**Start:** run cells from top to bottom (Shift+Enter), or use Run → Run All Cells. The first launch downloads the Python runtime and packages. Core analysis uses bundled training data; internet is needed for initial setup and interactive JavaScript. Each lab starts independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "folium==0.20.0", "plotly==6.3.0"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from fieldguide import setup_style, load_tables, summarize, TRAINING
setup_style()
print(TRAINING)

## Define the numerator before drawing a curve
Our fictional definition is **new suspected-syndrome case reports received by a district on a calendar day**. It is not a laboratory-confirmed case definition. We have report dates, not onset dates, so this is a **report-date curve**, not an onset-based epidemic curve. Real work would specify symptoms, eligibility, geography, onset and deduplication rules.
Read CDC's descriptive-epidemiology chapter for the distinction between counts, time and population comparisons [1]. The calculations here are an original teaching example.

In [ ]:
districts,reports,facilities=load_tables()
daily=reports.groupby("date").agg(cases=("cases_reported",lambda x:x.sum(min_count=1)),received=("reports_received","sum"),expected=("reports_expected","sum"))
fig,axes=plt.subplots(2,1,figsize=(11,6),sharex=True)
axes[0].bar(daily.index,daily.cases,width=1,color="#007f86")
axes[0].set(ylabel="New reported cases",title="SYNTHETIC · Report-date curve, not symptom onset")
axes[1].plot(daily.index,100*daily.received/daily.expected,marker="o",color="#db5b45")
axes[1].set(ylabel="Completeness (%)",ylim=(0,105),xlabel="Report date, November 2025 (UTC)")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

## Work the formula in the notebook
Seven-day observed reports per 100,000 = `sum(new reported cases in window) / resident population × 100,000`.
The label matters: it is an observed reporting measure. We do not divide by completeness to pretend we recovered unobserved cases. Districts can differ in access, care-seeking, definitions, testing and reporting delay.

In [ ]:
window=reports[reports.date.between("2025-11-22","2025-11-28")]
counts=window.groupby("district_id").cases_reported.sum(min_count=1)
analysis=districts.merge(counts.rename("cases"),on="district_id",validate="one_to_one")
analysis["reported_per_100k"]=analysis.cases/analysis.population*100000
display(analysis[["district","cases","population","reported_per_100k"]].round(1))
assert np.allclose(analysis.reported_per_100k,summarize().reported_per_100k)

## Sensitivity is not a confidence interval
Suppose the actual population were 20% lower or higher than our fixed estimate. Show the resulting range. These are **chosen scenario bounds**, not statistical confidence limits. They do not capture under-reporting or diagnostic uncertainty.

In [ ]:
a=analysis.sort_values("reported_per_100k").copy()
low=a.cases/(a.population*1.2)*100000
high=a.cases/(a.population*.8)*100000
fig,ax=plt.subplots(figsize=(10,5))
ax.errorbar(a.reported_per_100k,a.district,xerr=[a.reported_per_100k-low,high-a.reported_per_100k],fmt="o",capsize=4)
ax.set(xlabel="Observed reports per 100,000 · 7 days",title="SYNTHETIC · ±20% population scenario (not a confidence interval)")
plt.show()

In [ ]:
s=summarize()
fig,ax=plt.subplots()
points=ax.scatter(s.flood_pct,s.reported_per_100k,c=s.completeness_pct,cmap="viridis",s=85)
for row in s.itertuples():
    ax.annotate(row.district,(row.flood_pct,row.reported_per_100k),xytext=(4,4),textcoords="offset points",fontsize=8)
fig.colorbar(points,ax=ax,label="Reporting completeness (%)")
ax.set(xlabel="Fictional population exposed to flood (%)",ylabel="Reports per 100,000 · 7 days",title="An ecological pattern to investigate, not a causal estimate")
plt.show()

**Readout:** these associations were partly built into the synthetic generator. They are not evidence of a real exposure–disease relationship. Area-level associations also cannot establish an individual's exposure or risk (the ecological inference problem). Different boundary groupings can change apparent patterns.

## Practice and debrief

Identify two explanations for a falling report-date curve other than falling illness. Propose the additional fields needed for an onset curve. Explain why the denominator sensitivity bars cannot be called 95% confidence intervals.

## Sources and further learning

[1] [CDC: Describing epidemiologic data](https://www.cdc.gov/field-epi-manual/php/chapters/describing-epi-data.html). [2] [CDC: Designing analytic studies](https://www.cdc.gov/field-epi-manual/php/chapters/design-conduct-analyze-field-studies.html).